# Session 2: Agentic Loops and Task Decomposition
**CCAR-F task statements:** 1.1 (agentic loops) and 1.6 (task decomposition) · **Domain:** D1 Agentic Architecture (27%)
**Duration:** 60 minutes · **Extends:** Exercise 1, the Northwind support agent (Session 1 gave it one tool and one hand-written tool cycle)
**Pre-work:** API course, "Tool use with Claude" up to "Sending tool results" (from Session 1) and the stop_reason recap below.

By the end of this session you can:
1. Write correct agentic loop control flow: continue on `tool_use`, stop on `end_turn`.
2. Spot the three loop anti-patterns and debug a loop that stops too early.
3. Choose between prompt chaining and dynamic decomposition, and justify the choice.

| Min | Segment | What you do |
|---|---|---|
| 0-5 | Setup and recap | Check the environment, inspect `stop_reason` live |
| 5-15 | Essential concepts | Lifecycle, anti-patterns (read, then answer two questions) |
| 15-35 | Guided build | Add `get_customer`, implement `run_agent`, run it live |
| 35-45 | Debug | Reproduce and fix a broken loop |
| 45-52 | Decomposition | Classify five tasks: chain or dynamic? |
| 52-60 | Exam scenarios and wrap-up | Four scenario questions, homework briefing |

> **Instructor note.** Cells marked *Instructor note* are for the instructor. Answers sit inside collapsible `<details>` blocks so the learner can attempt first.

## 1. Setup and recap (5 min)

Same project as Session 1. Today's changes are already in `northwind/`:

* `backend.py` gained `CUSTOMERS`, `get_customer`, a third order, and two helpers for homework (`track_shipment`, `process_refund`).
* `common.py` now registers **two tools**: `lookup_order` and `get_customer`. Descriptions are still deliberately thin. Session 3 fixes that.
* `fake_client.py` and `test_loop_offline.py` let you test loop logic **without an API key**, using scripted responses.

In [2]:
import os, sys, json
sys.path.insert(0, "northwind")          # the notebook lives next to the northwind/ folder

assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in the environment that launched Jupyter, then restart the kernel."

from common import MODEL, SYSTEM, TOOLS, as_tool_result, client, run_tool, show
from backend import ORDERS, CUSTOMERS
print("Model:", MODEL)
print("Tools registered:", [t["name"] for t in TOOLS])

Model: claude-haiku-5-5
Tools registered: ['lookup_order', 'get_customer']


### Recap check: one request, read the control signal

Run this. Do **not** read the answer text yet. Look at `stop_reason` and the block types first.

In [3]:
messages = [{"role": "user", "content": "Where is order 12345?"}]
first = client.messages.create(model=MODEL, max_tokens=1024, system=SYSTEM, tools=TOOLS, messages=messages)
show(first)

stop_reason: tool_use
content blocks: 1
  [0] tool_use -> lookup_order
                  id=toolu_014sbQGW3QFgZQgR37DYArkp
                  input={'order_id': '12345'}


**Quick questions (answer out loud):**
1. What is `stop_reason` here, and what does your code have to do next?
2. The API is stateless. What must the *next* request contain?

<details><summary>Answers</summary>

1. `"tool_use"`. Your code must run the requested tool and send the result back. Claude never executes anything itself.
2. The full history: original user turn, the assistant turn containing the `tool_use` block, then a user turn containing the matching `tool_result`.
</details>

## 2. Essential concepts (10 min)

### The agentic loop lifecycle

```text
 user message
      |
      v
 +--> call Claude (system, tools, FULL history)
 |         |
 |     stop_reason?
 |      /        \
 |  "tool_use"   "end_turn"
 |     |              |
 |  append assistant   return the text
 |  turn (as produced)  <-- the only normal exit
 |     |
 |  run EVERY tool_use block
 |     |
 |  append ONE user turn with
 |  all tool_result blocks
 +-----+
```

**Model-driven vs pre-configured.** In an agentic loop *Claude* decides which tool comes next based on the previous result. A pre-configured sequence ("always call A, then B, then C") is a workflow. Both are valid. The exam tests whether you can tell which one a scenario needs.

### Three loop anti-patterns (task statement 1.1)

| Anti-pattern | Why it fails |
|---|---|
| Parsing natural language to decide termination ("if the text contains *done*...") | Wording varies run to run. `stop_reason` is the reliable signal. |
| An arbitrary iteration cap as the **main** stop | It cuts off legitimate work and lets runaway loops continue until the cap. A cap is a guard rail, not the exit. |
| Treating any assistant text as completion | Claude often writes text *and* calls a tool in the same response ("Let me check that for you"). |

**Two questions before we build:**
1. Why is a cap still worth keeping if `stop_reason` is the real exit?
2. In the lifecycle, why must the assistant's `tool_use` turn go into the history *before* the `tool_result` turn?

<details><summary>Answers</summary>

1. A bug or a misbehaving tool can keep Claude calling tools forever. The cap bounds cost and latency. When it triggers it should fail loudly (raise or escalate), never return a quiet "success".
2. The API checks that every `tool_result` answers a `tool_use` in the previous assistant turn. Without that turn the request is rejected, and even if it were not, Claude would have no record of what it asked for.
</details>

> **Exam habit (from Session 1):** find the root cause first. If a loop "sometimes returns half an answer", the fix is in the loop's control flow, not in the prompt.

## 3. Guided build (20 min)

### 3a. See why the loop must be dynamic

Customers do not know order numbers. Give the agent only an email address and look at what Claude asks for on the **first** call.

In [4]:
messages = [{"role": "user", "content": "Hi, I'm alex.rivera@example.com. Has my tent shipped?"}]
r1 = client.messages.create(model=MODEL, max_tokens=1024, system=SYSTEM, tools=TOOLS, messages=messages)
show(r1)

stop_reason: tool_use
content blocks: 1
  [0] tool_use -> get_customer
                  id=toolu_016zekGcumRcnduXTAuAuYWZ
                  input={'email': 'alex.rivera@example.com'}


**Discuss:** Claude asked for `get_customer`, not `lookup_order`. Why can't it call `lookup_order` in the same response?

<details><summary>Answer</summary>
It does not have an order ID yet. The ID is *inside* the `get_customer` result. The second call depends on the first result, so the sequence cannot be hard-coded in advance. That is what makes this an agentic loop rather than a fixed chain.
</details>

### 3b. Exercise: implement `run_agent`

Fill in the six TODOs. Rules:

* `stop_reason` decides what happens. No text parsing.
* Append the assistant turn **exactly as returned** (`response.content`, text blocks included).
* Run **every** `tool_use` block, then send **one** user turn with all results.
* `max_iterations` is a guard rail: exceeding it must `raise`.

Helpers you already have: `run_tool(name, input)` and `as_tool_result(block, output)`.

In [ ]:
def run_agent(user_message, client=client, max_iterations=15, verbose=True):
    # Returns (final_text, messages).
    messages = [{"role": "user", "content": user_message}]

    for step in range(1, max_iterations + 1):
        response = client.messages.create(
            model=MODEL, max_tokens=1024, system=SYSTEM, tools=TOOLS, messages=messages
        )
        if verbose:
            print(f"[step {step}] stop_reason={response.stop_reason}")

        # TODO 1: if response.stop_reason == "end_turn": return the joined text of the text blocks
        #         and the messages (including this final assistant turn)

        # TODO 2: if response.stop_reason == "tool_use": append the assistant turn to messages

        # TODO 3: run every tool_use block with run_tool and wrap each output with as_tool_result

        # TODO 4: append ONE user turn whose content is the list of tool_result blocks, then continue

        # TODO 5: any other stop_reason is neither "done" nor "keep going": raise RuntimeError
        raise NotImplementedError("finish the TODOs")

    # TODO 6: the guard rail was hit without end_turn: raise RuntimeError

### Self-check (offline, no API calls)

A live model only *sometimes* writes text before a tool call, so bugs that depend on it are intermittent. These checks use scripted responses so the same bug fails **every time**.

In [8]:
from test_loop_offline import check_loop, report
report(check_loop(run_agent))

PASS  continues past text + tool_use
PASS  returns the final end_turn text
PASS  history alternates user/assistant
PASS  tool_result matches the tool_use id
PASS  answers every tool_use block
PASS  ends on end_turn when no tools needed
PASS  guard rail raises when exceeded

7/7 checks passed


True

Stuck after two attempts? Run the next cell (it loads the reference solution into the cell), read it, then run it again and re-run the self-check.

> **Instructor note.** Let the learner struggle with TODO 3 and TODO 4. The usual mistakes are one `tool_result` turn *per* tool (instead of one turn with all results) and forgetting the assistant turn.

In [ ]:
# %load northwind/solutions/agent_loop.py

### 3c. Run it live

Three requests. Predict the number of steps for each *before* running.

In [9]:
for q in [
    "Hi! What are your store hours?",
    "Where is order 12345?",
    "Hi, I'm alex.rivera@example.com. Has my tent shipped?",
]:
    print("\n=== USER:", q)
    final, history = run_agent(q)
    print("AGENT:", final)
    print("steps:", sum(1 for m in history if m["role"] == "assistant"), "| messages in history:", len(history))


=== USER: Hi! What are your store hours?
AGENT: Hi there! I don't have access to our store hours, so I don't want to give you incorrect information. Our website's contact or help page should list current hours. Since we're an online store, you can also shop anytime.

If you need help with an order or a customer account, I can look that up. Just share your order number or customer ID.
steps: 1 | messages in history: 2

=== USER: Where is order 12345?
AGENT: Order 12345 has shipped. It's a Trailhead 2-person tent, and it's being carried by SwiftShip.

- **Tracking number:** SS-9081-4472
- **Estimated delivery:** October 2, 2026

Would you like help with anything else on this order?
steps: 2 | messages in history: 4

=== USER: Hi, I'm alex.rivera@example.com. Has my tent shipped?
AGENT: Hi Alex, yes, your tent has shipped!

**Order #12345: Trailhead 2-person tent** (1 × $249.00)
- Status: Shipped
- Carrier: SwiftShip
- Tracking number: SS-9081-4472
- Estimated delivery: October 2, 2026



**Why did you choose this approach?** Answer each in one sentence:
1. Why return after `end_turn` instead of checking whether the text "looks finished"?
2. Why does the loop send the *whole* `messages` list every time?
3. The email request took more steps than the order-ID request. Who decided that, your code or Claude?

<details><summary>Answers</summary>

1. `stop_reason` is a deterministic signal from the API. Text is free-form and varies.
2. The API is stateless. Anything not in the request does not exist for Claude.
3. Claude. The loop is the same code for all three requests. Only the model's decisions changed the path, which is the point of a model-driven loop.
</details>

## 4. Debug a broken loop (10 min)

A teammate wrote this loop. Customers report that "sometimes the agent just says it will look into it and stops." It passes a quick manual test with a simple question. Note the new `system` parameter: many production prompts tell Claude to narrate before calling tools.

In [10]:
SYSTEM_NARRATED = SYSTEM + " Before calling a tool, briefly tell the customer what you are about to do."

def run_agent_broken(user_message, client=client, max_iterations=10, verbose=True, system=SYSTEM):
    messages = [{"role": "user", "content": user_message}]
    for step in range(1, max_iterations + 1):
        response = client.messages.create(
            model=MODEL, max_tokens=1024, system=system, tools=TOOLS, messages=messages
        )
        if verbose:
            print(f"[step {step}] stop_reason={response.stop_reason}")

        for block in response.content:
            if block.type == "text":
                return block.text, messages

        messages.append({"role": "assistant", "content": response.content})
        results = []
        for block in response.content:
            if block.type == "tool_use":
                results.append(as_tool_result(block, run_tool(block.name, block.input)))
        messages.append({"role": "user", "content": results})
    raise RuntimeError("Hit the iteration cap")

**Step 1. Reproduce deterministically.**

In [11]:
report(check_loop(run_agent_broken))

FAIL  continues past text + tool_use
        hint: Expected 3 requests (get_customer, lookup_order, final answer) but the loop made 1. Did it stop because a text block appeared next to a tool_use?
FAIL  returns the final end_turn text
        hint: Return the text of the end_turn response, not the earlier preamble.
FAIL  history alternates user/assistant
        hint: Roles sent on the last request: ['user']
FAIL  tool_result matches the tool_use id
        hint: IndexError: list index out of range
PASS  answers every tool_use block
PASS  ends on end_turn when no tools needed
PASS  guard rail raises when exceeded

3/7 checks passed


False

**Step 2. Reproduce live.** Run it a few times. It may need several attempts, which is exactly why the offline check exists.

In [12]:
for attempt in range(3):
    final, history = run_agent_broken(
        "Hi, I'm alex.rivera@example.com. Has my tent shipped?", system=SYSTEM_NARRATED, verbose=True
    )
    print("AGENT SAID:", final, "\n")

[step 1] stop_reason=tool_use
AGENT SAID: I'll look up your account using your email address so I can find your tent order. 

[step 1] stop_reason=tool_use
AGENT SAID: I'll look up your customer account using your email address. 

[step 1] stop_reason=tool_use
AGENT SAID: I'll look up your customer account using your email address. 



**Step 3. Diagnose.** Before fixing, answer:
1. Which exact lines are the root cause?
2. Which of the three anti-patterns is this?
3. A teammate suggests adding "never write text before a tool call" to the system prompt. Why is that the wrong fix?

**Step 4. Fix it** in the copy below (change the control flow only) and re-run the self-check until all 7 pass.

In [13]:
def run_agent_fixed(user_message, client=client, max_iterations=10, verbose=True, system=SYSTEM):
    messages = [{"role": "user", "content": user_message}]
    for step in range(1, max_iterations + 1):
        response = client.messages.create(
            model=MODEL, max_tokens=1024, system=system, tools=TOOLS, messages=messages
        )
        if verbose:
            print(f"[step {step}] stop_reason={response.stop_reason}")

        for block in response.content:
            if block.type == "text":
                return block.text, messages

        messages.append({"role": "assistant", "content": response.content})
        results = []
        for block in response.content:
            if block.type == "tool_use":
                results.append(as_tool_result(block, run_tool(block.name, block.input)))
        messages.append({"role": "user", "content": results})
    raise RuntimeError("Hit the iteration cap")

report(check_loop(run_agent_fixed))

FAIL  continues past text + tool_use
        hint: Expected 3 requests (get_customer, lookup_order, final answer) but the loop made 1. Did it stop because a text block appeared next to a tool_use?
FAIL  returns the final end_turn text
        hint: Return the text of the end_turn response, not the earlier preamble.
FAIL  history alternates user/assistant
        hint: Roles sent on the last request: ['user']
FAIL  tool_result matches the tool_use id
        hint: IndexError: list index out of range
PASS  answers every tool_use block
PASS  ends on end_turn when no tools needed
PASS  guard rail raises when exceeded

3/7 checks passed


False

<details><summary>Diagnosis</summary>

1. The `for block in response.content: if block.type == "text": return ...` block. It returns the first text block it sees, even when the same response also contains a `tool_use` block.
2. "Treating assistant text as completion."
3. It treats a symptom, and prompt guidance has a non-zero failure rate. Narration before tool calls is normal, harmless behavior. The loop must be correct no matter what the model writes. (Exam habit: root cause first, deterministic control flow over prompt hope.)

**Fix:** delete the text-check block and branch on `response.stop_reason` as in `run_agent`.
</details>

### Name the anti-pattern

Which of the three is each snippet?

```python
# A
if "I've finished" in response.content[0].text or "anything else" in response.content[0].text:
    break

# B
for _ in range(3):                      # "three tool calls is plenty"
    response = call_claude(messages)
    ...handle tools...
return response                         # whatever we have after 3 rounds

# C
if any(block.type == "text" for block in response.content):
    return response
```

<details><summary>Answers</summary>

A: parsing natural language for termination. B: an arbitrary cap as the main stop (and it silently returns an unfinished answer). C: treating assistant text as completion, the bug you just fixed.
</details>

## 5. Task decomposition: chaining vs dynamic (7 min)

| | Prompt chaining | Dynamic adaptive decomposition |
|---|---|---|
| Steps known in advance? | Yes, same every time | No, findings decide the next step |
| Who decides the next step? | Your code (fixed sequence) | Claude, from intermediate results |
| Fits | Predictable multi-aspect reviews, fixed pipelines | Open-ended investigation |
| Example | Review each file for local issues, then one cross-file integration pass | "Add comprehensive tests to a legacy codebase": map the structure, find high-impact areas, build a prioritized plan, adapt as dependencies appear |

**Why per-file plus an integration pass?** One pass over 14 files dilutes attention and gives inconsistent depth. Per-file passes give uniform depth. A separate integration pass catches cross-file problems like data-flow mismatches.

### Exercise: classify

Fill in `answers` with `"chain"` or `"dynamic"`, then run the check.

1. Review a 14-file pull request for style, security and test coverage, the same way every time.
2. "Add comprehensive tests to this legacy codebase."
3. Customer gives only an email and asks about their latest order (today's Northwind agent).
4. Every night: extract fields from each invoice, validate them, then produce a summary report. Same three steps every time.
5. "Why did checkout latency regress after last week's release?"

In [ ]:
answers = {
    1: "",   # "chain" or "dynamic"
    2: "",
    3: "dynamic",
    4: "",
    5: "",
}

KEY = {
    1: ("chain",   "Fixed aspects, known in advance: per-file passes + one integration pass."),
    2: ("dynamic", "You cannot plan the test work until you have mapped the code. Findings reshape the plan."),
    3: ("dynamic", "The second tool call depends on data returned by the first (the order IDs)."),
    4: ("chain",   "Three known steps in a fixed order, so a fixed sequence is simpler and cheaper."),
    5: ("dynamic", "Each finding (a slow query, a config change) decides where to look next."),
}

for i, (want, why) in KEY.items():
    got = answers[i].strip().lower()
    mark = "OK  " if got == want else "MISS"
    print(f"{mark} {i}: you said {got or '-':8} | answer: {want:8} | {why}")

MISS 1: you said -        | answer: chain    | Fixed aspects, known in advance: per-file passes + one integration pass.
MISS 2: you said -        | answer: dynamic  | You cannot plan the test work until you have mapped the code. Findings reshape the plan.
OK   3: you said dynamic  | answer: dynamic  | The second tool call depends on data returned by the first (the order IDs).
MISS 4: you said -        | answer: chain    | Three known steps in a fixed order, so a fixed sequence is simpler and cheaper.
MISS 5: you said -        | answer: dynamic  | Each finding (a slow query, a config change) decides where to look next.


**Why did you choose this approach?** For #3, you could also hard-code `get_customer -> lookup_order`. What breaks when a customer gives an order number instead of an email, or has three orders?

<details><summary>Answer</summary>
A hard-coded sequence either wastes a call (looking up a customer you do not need) or fails on the variants. The model-driven loop handles the variants with the same code. Use a fixed chain only when the steps really are fixed.
</details>

## 6. Exam-style scenarios (5 min)

**Scenario: Customer Support Resolution Agent** (the exam's Scenario 1: D1, D2, D5). Take two or three questions in class and set the rest as homework. Reason aloud: root cause first, then the proportionate fix.

---
**Q1 (1.1).** Your support agent loop ends as soon as a response contains a text block. In production, some customers receive "Let me check that for you" as the final answer, with no result. Which change fixes the root cause?

A. Add "do not write text before calling a tool" to the system prompt.
B. Continue the loop while `stop_reason == "tool_use"` and end only on `"end_turn"`.
C. Lower the iteration cap to three so the loop cannot run long.
D. End the loop when the text contains phrases such as "I have looked up" or "your order".

<details><summary>Answer and reasoning</summary>

**B.** Completion is signalled by `stop_reason`, not by whether text is present. Text next to `tool_use` is normal.
A relies on prompt compliance (non-zero failure rate) for something the control flow should guarantee. C is an arbitrary cap as the main stop and would make the truncation *worse*. D is natural-language parsing, which breaks when wording changes.
</details>

---
**Q2 (1.1).** After running a requested tool, your code sends the next request with the original user message plus a user turn containing the `tool_result`. The API rejects it. What is missing?

A. The assistant turn containing the `tool_use` block, placed before the `tool_result` turn.
B. A larger `max_tokens` so the response is not cut off.
C. `tool_choice: {"type": "any"}` so Claude must use the result.
D. A retry with exponential backoff, since the failure is transient.

<details><summary>Answer and reasoning</summary>

**A.** Each `tool_result` must answer a `tool_use` in the preceding assistant turn, and the history must alternate user/assistant.
B has nothing to do with a malformed request. C controls which tool Claude picks, not history structure. D retries a request that will fail identically every time. This is a validation-style failure, not a transient one.
</details>

---
**Q3 (1.6).** A team wants consistent, thorough automated review of 14-file pull requests covering local issues and cross-file problems such as data-flow mismatches. A single prompt over all files gives uneven depth and misses cross-file issues. Which design fits best?

A. Keep one prompt but raise `max_tokens`.
B. Let Claude decide dynamically how to split the review each time.
C. Ask Claude to review the whole PR twice and keep the findings that appear both times.
D. Analyze each file in its own pass for local issues, then run a separate integration pass for cross-file issues.

<details><summary>Answer and reasoning</summary>

**D.** The aspects are known in advance, so a fixed chain is predictable and gives uniform per-file depth, with a dedicated pass for cross-file concerns.
A does not address attention dilution. B adds unpredictability where the steps are already known (proportionate solution: do not use adaptive decomposition for a fixed task). C is self-repetition, not decomposition, and does not fix cross-file blindness.
</details>

---
**Q4 (1.6).** A developer asks Claude to "add comprehensive tests to our legacy codebase." Nobody knows which modules are risky or how they depend on each other. Which approach is best?

A. Generate tests for every file in alphabetical order using one fixed prompt per file.
B. Run Claude Code with an `--auto-plan` flag that decomposes any task automatically.
C. Map the structure first, identify high-impact areas, write a prioritized plan, and revise the plan as dependencies are discovered.
D. Ask for all tests in a single response to keep context unified.

<details><summary>Answer and reasoning</summary>

**C.** Open-ended work where findings shape later steps calls for dynamic, adaptive decomposition.
A is a fixed chain that ignores impact and dependencies. B is a plausible-sounding feature that does not exist; reject options that name features you cannot verify in the docs. D overloads one response and cannot adapt.
</details>

## 7. Wrap-up and homework (3 min)

Persist your loop so every later session can import it. (If you skipped the exercise, this copies the reference solution.)

In [16]:
import shutil, pathlib
target = pathlib.Path("northwind/agent_loop.py")
if not target.exists():
    shutil.copy("northwind/solutions/agent_loop.py", target)
    print("Copied the reference solution to", target)
else:
    print(target, "already exists; keeping yours.")

northwind\agent_loop.py already exists; keeping yours.


### Homework (from the curriculum)

* [ ] API course: finish **Tool use with Claude** (multi-turn conversations with tools, multiple tools) and the **Agents and workflows** unit.
* [ ] **Start Exercise 1 (continued):** define **3 to 4 tools, two overlapping on purpose**, and run them through `run_agent`.
  * `backend.py` already has `track_shipment` and `process_refund`. You write the *definitions* in `common.py` (name, description, `input_schema`) and register them in `TOOL_FUNCTIONS`.
  * `track_shipment` deliberately overlaps with `lookup_order` (which also returns carrier and tracking). Keep the descriptions short and natural; Session 3 is where you measure what that overlap costs.
  * Run: "Where is order 12345?" and "Refund 12399, the stove arrived broken." Log which tools Claude picked and how many steps each took.
* [ ] Re-do exam Q2 and Q4 in a few days without the answers.

### Next session
**Session 3: Tool interface design and tool distribution (2.1, 2.3).** Bring your homework tool definitions and your log of which tool Claude chose.

> **Instructor note.** Before Session 3, skim the learner's logs: any request where Claude picked `lookup_order` for a shipping question (or the reverse) is the opening example.